# 03 — Step 2: Attention Entropy Analysis

**Goal**: For candidate heads, compute attention entropy on mirror vs non-mirror images.

Reflection heads should show **lower entropy** on mirror images (more focused attention pattern).

$$H(h, l) = -\sum_{ij} A \log A$$

Combined score: $\text{HIES} = S \times (H_{\text{nonmirror}} - H_{\text{mirror}}) / H_{\max}$

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from scripts.config import (
    EXPERIMENTS_DIR, MIRROR_PROMPTS, NON_MIRROR_PROMPTS, SEEDS,
    PROMPT_OBJECTS, MIRROR_DIR, NON_MIRROR_DIR,
)
from scripts.model_loader import load_flux_pipeline
from scripts.roi import get_object_and_reflection_rois
from scripts.generate import generate_with_full_attention
from scripts.attention_extraction import AttentionData
from scripts.metrics import compute_entropy_from_data, compute_hies
from scripts.visualization import plot_entropy_comparison, plot_hies_scores

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP2_DIR = EXPERIMENTS_DIR / "step2_entropy"
STEP2_DIR.mkdir(parents=True, exist_ok=True)

# Load Step 1 results
with open(STEP1_DIR / "step1_results.pkl", "rb") as f:
    step1 = pickle.load(f)
candidates = step1["candidates"]
selectivity = step1["selectivity_matrix"]

candidate_blocks = list(set(b for b, h, s in candidates))
print(f"Candidate blocks for full extraction: {sorted(candidate_blocks)}")
print(f"Top candidates: {[(b, h) for b, h, _ in candidates]}")
print(f"Using CLIPSeg ROIs per image")

In [ ]:
pipe = load_flux_pipeline(quantize_nf4=True, cpu_offload=True)
print("Model loaded.")

In [ ]:
# Extract full attention maps for candidate blocks — mirror images
# Use a subset to save time (first 4 prompts × first seed)
print("=== Mirror — Full Attention Extraction ===")
mirror_full_data = []

for i in range(min(4, len(MIRROR_PROMPTS))):
    seed = SEEDS[0]
    obj_name = PROMPT_OBJECTS[i]
    print(f"  Prompt {i} ({obj_name}), seed {seed}...", end=" ")
    # Load pre-generated image for CLIPSeg ROI
    img_path = MIRROR_DIR / f"prompt{i:02d}_seed{seed}.png"
    image = Image.open(img_path)
    obj_roi, ref_roi = get_object_and_reflection_rois(image, obj_name)
    attn_data = AttentionData()
    image, attn_data = generate_with_full_attention(
        pipe, MIRROR_PROMPTS[i], seed, obj_roi, ref_roi,
        candidate_blocks=candidate_blocks, attention_data=attn_data,
    )
    mirror_full_data.append(attn_data)
    print(f"done ({len(attn_data.full_maps)} maps, obj={obj_roi.size}t ref={ref_roi.size}t)")

print(f"Collected {len(mirror_full_data)} mirror samples with full maps")

In [ ]:
# Extract full attention maps — non-mirror images
print("=== Non-Mirror — Full Attention Extraction ===")
nonmirror_full_data = []

for i in range(min(4, len(NON_MIRROR_PROMPTS))):
    seed = SEEDS[0]
    obj_name = PROMPT_OBJECTS[i]
    print(f"  Prompt {i} ({obj_name}), seed {seed}...", end=" ")
    # Load pre-generated image for CLIPSeg ROI
    img_path = NON_MIRROR_DIR / f"prompt{i:02d}_seed{seed}.png"
    image = Image.open(img_path)
    obj_roi, ref_roi = get_object_and_reflection_rois(image, obj_name)
    attn_data = AttentionData()
    image, attn_data = generate_with_full_attention(
        pipe, NON_MIRROR_PROMPTS[i], seed, obj_roi, ref_roi,
        candidate_blocks=candidate_blocks, attention_data=attn_data,
    )
    nonmirror_full_data.append(attn_data)
    print(f"done ({len(attn_data.full_maps)} maps, obj={obj_roi.size}t ref={ref_roi.size}t)")

print(f"Collected {len(nonmirror_full_data)} non-mirror samples with full maps")

In [ ]:
# Compute entropy
mirror_entropy = compute_entropy_from_data(mirror_full_data, candidate_blocks)
nonmirror_entropy = compute_entropy_from_data(nonmirror_full_data, candidate_blocks)

print(f"{'Head':<10} {'H_mirror':<12} {'H_nonmirror':<14} {'Diff':<10}")
print("-" * 46)
for b, h, s in candidates:
    key = (b, h)
    if key in mirror_entropy and key in nonmirror_entropy:
        hm = mirror_entropy[key]
        hn = nonmirror_entropy[key]
        print(f"B{b}H{h:<5} {hm:<12.4f} {hn:<14.4f} {hn - hm:<10.4f}")

In [ ]:
# Compute HIES scores
hies_scores = {}
for b, h, s in candidates:
    key = (b, h)
    if key in mirror_entropy and key in nonmirror_entropy:
        hies = compute_hies(s, mirror_entropy[key], nonmirror_entropy[key])
        hies_scores[key] = hies

print("\nHIES Rankings:")
sorted_hies = sorted(hies_scores.items(), key=lambda x: x[1], reverse=True)
for rank, ((b, h), score) in enumerate(sorted_hies, 1):
    print(f"  {rank}. B{b}H{h}: HIES = {score:.6f}")

In [ ]:
# Visualize
fig = plot_entropy_comparison(mirror_entropy, nonmirror_entropy, candidates)
fig.savefig(STEP2_DIR / "entropy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

fig = plot_hies_scores(hies_scores)
fig.savefig(STEP2_DIR / "hies_scores.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save results
results = {
    "mirror_entropy": mirror_entropy,
    "nonmirror_entropy": nonmirror_entropy,
    "hies_scores": hies_scores,
    "ranked_hies": sorted_hies,
}
with open(STEP2_DIR / "step2_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP2_DIR}")